# Engram P2.6: Gemma as the fact writer, two forms, fresh sentences

Gemma 3 4B writes facts for 200 new sentences twice: in the free line form, and in the forced form where the relation can only come from the list. Benchmark data only. About 15 to 20 minutes, mostly setup.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Save Version > Save & Run All. When it finishes, open the saved version's **Output** and download the two `*_replies.json` files, or use the links at the end of this page.


In [ ]:
COMMIT = "77fa6d4"
import os, subprocess, sys, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
subprocess.run(f"cd Engram && git checkout -q {COMMIT}", shell=True, check=True)
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="4")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try: urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception: time.sleep(2)
else: raise SystemExit("Ollama did not start. See ollama.log in the Output.")
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
LOG = f"{SAVE}/p26b_run.log"
for form in ("free", "json"):
    p = subprocess.run([sys.executable, "-u", "kit_tools/p26_gemma.py", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE,
                        "--sample", "results/p26b/sample.jsonl", "--form", form], cwd=f"{WORK}/Engram", stdout=open(LOG, "a"), stderr=subprocess.STDOUT)
    if p.returncode: print(open(LOG).read()[-1500:]); raise SystemExit(f"The {form} run did not finish. Send Claude p26b_run.log.")
print(open(LOG).read()[-1500:])


In [ ]:
os.chdir("/kaggle/working")
from IPython.display import FileLink, display
for f in ("p26b_free_replies.json", "p26b_json_replies.json", "p26b_run.log"):
    display(FileLink(f"engram_results/{f}"))


Send Claude `p26b_free_replies.json`, `p26b_json_replies.json` and `p26b_run.log`.
